# Bitwen — Claims 2 & 3 via hosted inference (A-Bitwen + neural baseline)

Self-contained reproduction of the **agentic** claims of *Learning Randomized Reductions* (Bitwen, ICML 2026) for the ICML 2026 Agent Reproduction Challenge.

- **Claim 2:** Agentic Bitwen finds RSRs for 64/80 functions (80%).
- **Claim 3:** Agentic Bitwen outperforms a pure-neural baseline on discovery + verification.

**Backend substitution (permitted by the challenge):** the paper uses GPT-OSS-120B / Claude (proprietary). We point Bitwen's upstream `OpenAIAgent` at an **open model via an OpenAI-compatible endpoint** — **no code changes, no GPU, no vLLM**.

**No GPU needed** — inference is hosted (a free Colab **CPU** runtime works). After adding a key in cell 2, **Runtime → Run all**.


## 1. Environment — clone Bitwen, install agentic + LR deps, Node.js (for the MCP tool)


In [ ]:
import sys; print("python", sys.version.split()[0], "(no GPU required — inference is hosted)")


In [ ]:
!rm -rf learning-randomized-reductions
!git clone --depth 1 https://github.com/ferhaterata/learning-randomized-reductions.git
%cd /content/learning-randomized-reductions


In [ ]:
# LR-path + agentic deps (skip PySR/Julia, GPLearn, Gurobi license). No vLLM.
!pip install -q -e . --no-deps
!pip install -q numpy sympy scipy pandas scikit-learn joblib tqdm func-timeout pulp z3-solver pycparser python-dotenv gurobipy
!pip install -q "strands-agents[openai]==1.20.0" "strands-agents-tools==0.2.18" "mcp==1.25.0" openai
!sudo apt-get update -qq && sudo apt-get install -y -qq nodejs npm


## 2. Pick a working endpoint — tries HF Inference Providers, then Groq
Add keys as **Colab Secrets** (🗝 left panel) and enable notebook access:
- `HF_TOKEN` — your HF token (https://huggingface.co/settings/tokens). **Primary.**
- `GROQ_API_KEY` — free key from https://console.groq.com/keys. **Fallback when HF is down.**

The cell pings each provider's model candidates with retry (504/5xx are transient), auto-selects the first that responds, and verifies tool-calling. If HF is erroring, add the Groq key and re-run — it'll use Groq automatically.


In [ ]:
import os, time
from openai import OpenAI

def get_secret(name, required=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    if not required:
        return os.environ.get(name)
    import getpass
    return getpass.getpass(f"Paste {name}: ")

HF_TOKEN  = get_secret("HF_TOKEN", required=True)
GROQ_KEY  = get_secret("GROQ_API_KEY", required=False)

PROVIDERS = []
if HF_TOKEN:
    PROVIDERS.append(("hf", "https://router.huggingface.co/v1", HF_TOKEN,
                      ["openai/gpt-oss-120b",  # paper's exact model — REQUIRED (others skip Bitwen's answer format)
                     "Qwen/Qwen2.5-72B-Instruct", "meta-llama/Llama-3.3-70B-Instruct",
                       "Qwen/Qwen2.5-32B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]))
if GROQ_KEY:
    PROVIDERS.append(("groq", "https://api.groq.com/openai/v1", GROQ_KEY,
                      ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]))

def chat(cli, model, messages, tools=None, tries=4, delay=10, **kw):
    last = None
    for i in range(tries):
        try:
            return cli.chat.completions.create(model=model, messages=messages, tools=tools, **kw)
        except Exception as e:
            last = e
            code = getattr(getattr(e, "response", None), "status_code", None)
            print(f"    [{i+1}/{tries}] {model}: HTTP {code} {repr(e)[:70]}")
            if code in (401, 403, 404):
                break  # auth/model error — retry won't help
            time.sleep(delay)
    raise last

PROVIDER = BASE_URL = API_KEY = MODEL = None
for name, url, key, models in PROVIDERS:
    print(f"== provider {name} ({url}) ==")
    cli = OpenAI(base_url=url, api_key=key)
    for m in models:
        try:
            r = chat(cli, m, [{"role": "user", "content": "reply: ok"}], max_tokens=5)
            print(f"  ok: {m} -> {r.choices[0].message.content!r}")
            PROVIDER, BASE_URL, API_KEY, MODEL = name, url, key, m
            break
        except Exception as e:
            print(f"  fail: {m}: {repr(e)[:90]}")
    if MODEL:
        break

if not MODEL:
    raise SystemExit("No provider responded. HF router may be down — add a free GROQ_API_KEY "
                     "(https://console.groq.com/keys) as a Colab Secret and re-run this cell.")
print(f"\nUSING provider={PROVIDER} model={MODEL}\n  base_url={BASE_URL}")

cli = OpenAI(base_url=BASE_URL, api_key=API_KEY)
r = chat(cli, MODEL, [{"role": "user", "content": "call add(2,2)"}],
         tools=[{"type": "function", "function": {"name": "add",
                 "parameters": {"type": "object",
                                "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
                                "required": ["a", "b"]}}}], max_tokens=50)
print("tool_calls present:", r.choices[0].message.tool_calls is not None,
      "(False => agent loop won't work; re-run to try another model)")


## 3. Claim 2 — A-Bitwen over all 80 RSR-Bench functions
Uses the provider/model/key chosen in cell 2.


In [ ]:
import os; os.makedirs("outputs/abitween", exist_ok=True)
for mod in ["bitween.evaluation.evaluation_rsr_bench_agentic_paper",
            "bitween.evaluation.evaluation_rsr_bench_agentic_paper_extended"]:
    print(">>> A-Bitwen:", mod, flush=True)
    rc = os.system(f'''python -m {mod} --agent_type openai --model_id "{MODEL}" '''
                   f'''--base_url {BASE_URL} --api_key "{API_KEY}" '''
                   f'''--max_tokens 32000 --timeout_sec 1800 --res_dir outputs/abitween '''
                   f'''--custom_tools infer_property_tool symbolic_verify_tool''')
    print(f"{mod} -> exit {rc}")


## 4. Claim 3 — neural baseline (same model, Bitwen tools OFF)


In [ ]:
import os; os.makedirs("outputs/neural", exist_ok=True)
for mod in ["bitween.evaluation.evaluation_rsr_bench_agentic_paper",
            "bitwen.evaluation.evaluation_rsr_bench_agentic_paper_extended"]:
    print(">>> Neural baseline:", mod, flush=True)
    rc = os.system(f'''python -m {mod} --agent_type openai --model_id "{MODEL}" '''
                   f'''--base_url {BASE_URL} --api_key "{API_KEY}" '''
                   f'''--max_tokens 32000 --timeout_sec 1800 --res_dir outputs/neural '''
                   f'''--custom_tools''')
    print(f"{mod} -> exit {rc}")


## 5. Compare A-Bitwen vs neural (Claim 3) + package results


In [ ]:
import os, re, glob
def summarize(d):
    cov=v=uv=tok=0; n=0
    for f in glob.glob(f"{d}/*.txt"):
        n+=1; t=open(f,errors="replace").read()
        mv=re.search(r"^Verified \((\d+)\):", t, re.M); vv=int(mv.group(1)) if mv else 0
        mu=re.search(r"^Unverified \((\d+)\):", t, re.M); uuu=int(mu.group(1)) if mu else 0
        if vv==0 and "proved: True" in t: vv=t.count("proved: True")
        v+=vv; uv+=uuu
        if vv>0: cov+=1
        mt=re.search(r"tokens[:=]\s*([\d.eE]+)", t, re.I)
        if mt:
            try: tok+=int(float(mt.group(1)))
            except: pass
    cand=v+uv
    return dict(funcs=n,covered=cov,cov_pct=100*cov/n if n else 0,verified=v,unverified=uv,
                acc=(v/cand) if cand else 0,tokens=tok)
a=summarize("outputs/abitween"); n=summarize("outputs/neural")
print("="*64); print(f"Claim 3: Agentic vs Neural ({PROVIDER}/{MODEL})"); print("="*64)
for k,l,fmt in [("funcs","functions",'{:d}'),("covered","covered",'{:d}'),("cov_pct","coverage %",'{:.1f}'),
                ("verified","verified ids",'{:d}'),("acc","verify accuracy",'{:.1%}'),("tokens","tokens",'{:d}')]:
    print(f"  {l:20s} {fmt.format(a[k]):>12} {fmt.format(n[k]):>12}")
print("C3 outcome:", "AGENTIC>=NEURAL" if a['covered']>=n['covered'] and a['acc']>=n['acc'] else "mixed")


In [ ]:
!cd outputs && zip -r -q /content/bitwen_c2_c3_results.zip abitween neural
from google.colab import files; files.download("/content/bitwen_c2_c3_results.zip")


## 6. Send results back
Download `bitwen_c2_c3_results.zip` (auto-triggered) and share it — or paste the "Claim 3" table. It will be folded into the logbook at https://huggingface.co/spaces/DineshAI/hCAEcqig2C and republished.

**Notes**
- **HF router errors (504/5xx):** add a free `GROQ_API_KEY` Colab Secret (https://console.groq.com/keys); the notebook then auto-uses Groq (Llama-3.3-70B, tool-calling, fast).
- **Tool-calling is mandatory.** If cell 2 reports `tool_calls present: False`, re-run to land on another model.
- **Cost/limits:** HF bills per-token; Groq has a generous free tier (rate-limited). On free tiers the 80×2 sweep may take a few hours due to rate limits.
- **Coverage caveat:** an open 70–72B is below the paper's Opus/GPT-OSS-120B, so absolute coverage may land under 80%; we report the real number. The Claim-3 *mechanism* (agentic > neural) is what we test and should hold.
